# Regressão Logística para Predição de Alta ou Baixa do Preço das Ações SLCE3

Este notebook implementa modelos de regressão logística para prever se o preço de fechamento das ações SLCE3 irá subir ou não em diferentes horizontes temporais (3, 7, 15 e 30 dias).

## 1. Importação de Bibliotecas e Configuração Inicial

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


## 2. Carregamento e Pré-processamento dos Dados

In [2]:
# Carregamento do dataset
df = pd.read_csv('SLCE3.csv')

print("Informações do dataset:")
print(f"Shape: {df.shape}")
print(f"Colunas: {list(df.columns)}")
print("\nPrimeiras 5 linhas:")
print(df.head())

Informações do dataset:
Shape: (1738, 6)
Colunas: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

Primeiras 5 linhas:
         Date     Close      High       Low      Open   Volume
0  2018-01-02  3.077934  3.103537  2.972182  2.972182  1227908
1  2018-01-03  3.104650  3.215968  3.067915  3.077933  2494536
2  2018-01-04  3.092405  3.198157  3.055670  3.104650  2957724
3  2018-01-05  3.150290  3.166988  3.069028  3.138046  2140248
4  2018-01-08  3.133592  3.183685  3.112442  3.150290  1613172


In [3]:
# Criando colunas de valores futuros para diferentes horizontes temporais
df['Close_3d_fut'] = df['Close'].shift(-3)
df['Close_7d_fut'] = df['Close'].shift(-7)
df['Close_15d_fut'] = df['Close'].shift(-15)
df['Close_30d_fut'] = df['Close'].shift(-30)

# Removendo linhas com valores NaN
df_process = df.dropna(subset=['Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut']).copy()

# Convertendo coluna de data e definindo como índice
df_process['Date'] = pd.to_datetime(df_process['Date'])
df_process.set_index('Date', inplace=True)

print(f"Dataset após processamento: {df_process.shape}")
print(f"Período: {df_process.index[0]} até {df_process.index[-1]}")

Dataset após processamento: (1708, 9)
Período: 2018-01-02 00:00:00 até 2024-11-12 00:00:00


In [4]:
# Criando variável dicotômica para períodos específicos
df_process['dummy_period'] = 0
df_process.loc[df_process.index >= pd.to_datetime('2021-05-05'), 'dummy_period'] = 1
df_process.loc[df_process.index >= pd.to_datetime('2022-02-07'), 'dummy_period'] = 2

# Criando variáveis target binárias para cada horizonte temporal
# 1 = preço futuro > preço atual (alta), 0 = preço futuro <= preço atual (baixa/estável)
df_process['target_3d'] = (df_process['Close_3d_fut'] > df_process['Close']).astype(int)
df_process['target_7d'] = (df_process['Close_7d_fut'] > df_process['Close']).astype(int)
df_process['target_15d'] = (df_process['Close_15d_fut'] > df_process['Close']).astype(int)
df_process['target_30d'] = (df_process['Close_30d_fut'] > df_process['Close']).astype(int)

print("Distribuição das classes (1=Alta, 0=Baixa/Estável):")
for horizon in ['3d', '7d', '15d', '30d']:
    target_col = f'target_{horizon}'
    distribution = df_process[target_col].value_counts()
    percentage = df_process[target_col].mean()
    print(f"{horizon}: {distribution.to_dict()} (Taxa de alta: {percentage:.1%})")
df_process.to_csv('SLCE3_processed.csv')

Distribuição das classes (1=Alta, 0=Baixa/Estável):
3d: {1: 870, 0: 838} (Taxa de alta: 50.9%)
7d: {1: 904, 0: 804} (Taxa de alta: 52.9%)
15d: {1: 914, 0: 794} (Taxa de alta: 53.5%)
30d: {1: 955, 0: 753} (Taxa de alta: 55.9%)


## 3. Modelos de Regressão Logística com Variável Dummy Period

In [5]:
# Preparação das features com dummy_period (sem aplicar transformação global para evitar vazamento)
features_with_dummy = ['Low', 'High', 'Open', 'dummy_period']
X_with_dummy = df_process[features_with_dummy].copy()

# Definição do preprocessor (apenas definição; o fit ocorrerá no treino dentro do Pipeline)
preprocessor_with_dummy = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Low', 'High', 'Open']),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ['dummy_period'])
    ],
    remainder='drop'
)

print("Features (com dummy_period + One-Hot) definidas:", features_with_dummy)

Features (com dummy_period + One-Hot) definidas: ['Low', 'High', 'Open', 'dummy_period']


In [6]:
# Treinamento dos modelos com dummy_period usando Pipeline para evitar vazamento
horizons = ['3d', '7d', '15d', '30d']
results_with_dummy = {}

print("="*80)
print("MODELOS DE REGRESSÃO LOGÍSTICA COM DUMMY_PERIOD (ONE-HOT ENCODED)")
print("="*80)

# Define Pipeline (preprocessa e treina dentro do fit, apenas com dados de treino)
pipeline_with_dummy = Pipeline(steps=[
    ('preprocess', preprocessor_with_dummy),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

for horizon in horizons:
    print(f"\n{'-'*50}")
    print(f"MODELO PARA HORIZONTE DE {horizon[:-1]} DIAS")
    print(f"{'-'*50}")
    
    # Preparação dos dados para este horizonte
    target_col = f'target_{horizon}'
    y = df_process[target_col].copy()
    
    # Removendo linhas com valores NaN
    mask = ~(X_with_dummy.isnull().any(axis=1) | y.isnull())
    X_clean = X_with_dummy[mask]
    y_clean = y[mask]
    
    # Divisão treino/teste (mantém sua estratégia atual; se quiser evitar embaralhar, troque por split temporal)
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
    )
    
    # Treinamento do pipeline (fit apenas no treino)
    pipeline_with_dummy.fit(X_train, y_train)
    
    # Predições
    y_pred = pipeline_with_dummy.predict(X_test)
    
    # Cálculo das métricas específicas solicitadas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    # Armazenando resultados
    results_with_dummy[horizon] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'support_total': len(y_test)
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Support: {len(y_test)} amostras")

MODELOS DE REGRESSÃO LOGÍSTICA COM DUMMY_PERIOD (ONE-HOT ENCODED)

--------------------------------------------------
MODELO PARA HORIZONTE DE 3 DIAS
--------------------------------------------------
Accuracy: 0.5263
Precision: 0.5273
Recall: 0.5263
F1-Score: 0.5257
Support: 342 amostras

--------------------------------------------------
MODELO PARA HORIZONTE DE 7 DIAS
--------------------------------------------------
Accuracy: 0.5731
Precision: 0.5761
Recall: 0.5731
F1-Score: 0.5733
Support: 342 amostras

--------------------------------------------------
MODELO PARA HORIZONTE DE 15 DIAS
--------------------------------------------------
Accuracy: 0.5965
Precision: 0.5945
Recall: 0.5965
F1-Score: 0.5938
Support: 342 amostras

--------------------------------------------------
MODELO PARA HORIZONTE DE 30 DIAS
--------------------------------------------------
Accuracy: 0.6023
Precision: 0.6048
Recall: 0.6023
F1-Score: 0.6032
Support: 342 amostras


## 4. Modelos de Regressão Logística sem Variável Dummy Period

In [7]:
# Preparação das features sem dummy_period (sem aplicar transformação global para evitar vazamento)
features_without_dummy = ['Low', 'High', 'Open']
X_without_dummy = df_process[features_without_dummy].copy()

# Definição do preprocessor para o caso sem dummy (fit ocorrerá apenas no treino)
preprocessor_without_dummy = ColumnTransformer(
    transformers=[('num', StandardScaler(), ['Low', 'High', 'Open'])],
    remainder='drop'
)

print("Features utilizadas (sem dummy_period):", features_without_dummy)
print("Shape dos dados:", X_without_dummy.shape)

Features utilizadas (sem dummy_period): ['Low', 'High', 'Open']
Shape dos dados: (1708, 3)


In [8]:
# Treinamento dos modelos sem dummy_period
results_without_dummy = {}

print("="*80)
print("MODELOS DE REGRESSÃO LOGÍSTICA SEM DUMMY_PERIOD")
print("="*80)

# Define Pipeline para o caso sem dummy
pipeline_without_dummy = Pipeline(steps=[
    ('preprocess', preprocessor_without_dummy),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

for horizon in horizons:
    print(f"\n{'-'*50}")
    print(f"MODELO PARA HORIZONTE DE {horizon[:-1]} DIAS")
    print(f"{'-'*50}")
    
    # Preparação dos dados para este horizonte
    target_col = f'target_{horizon}'
    y = df_process[target_col].copy()
    
    # Removendo linhas com valores NaN
    mask = ~(X_without_dummy.isnull().any(axis=1) | y.isnull())
    X_clean = X_without_dummy[mask]
    y_clean = y[mask]
    
    # Divisão treino/teste
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
    )
    
    # Treinamento do pipeline
    pipeline_without_dummy.fit(X_train, y_train)
    
    # Predições
    y_pred = pipeline_without_dummy.predict(X_test)
    
    # Cálculo das métricas específicas solicitadas
    accuracy = accuracy_score(y_test, y_pred)
    precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    # Métricas adicionais mais específicas para classificação binária
    precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    # Métricas específicas para a classe positiva (Alta)
    precision_alta = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    recall_alta = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1_alta = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    
    # Armazenando resultados
    results_without_dummy[horizon] = {
        'accuracy': accuracy,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_score_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_score_macro': f1_macro,
        'precision_alta': precision_alta,
        'recall_alta': recall_alta,
        'f1_score_alta': f1_alta,
        'support_total': len(y_test)
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision (weighted): {precision_weighted:.4f}")
    print(f"Recall (weighted): {recall_weighted:.4f}")
    print(f"F1-Score (weighted): {f1_weighted:.4f}")
    print(f"Precision (macro): {precision_macro:.4f}")
    print(f"Recall (macro): {recall_macro:.4f}")
    print(f"F1-Score (macro): {f1_macro:.4f}")
    print(f"Precision (Alta): {precision_alta:.4f}")
    print(f"Recall (Alta): {recall_alta:.4f}")
    print(f"F1-Score (Alta): {f1_alta:.4f}")
    print(f"Support: {len(y_test)} amostras")
    
    # Relatório detalhado por classe
    print("\nRelatório detalhado por classe:")
    print(classification_report(y_test, y_pred, target_names=['Baixa/Estável', 'Alta']))

MODELOS DE REGRESSÃO LOGÍSTICA SEM DUMMY_PERIOD

--------------------------------------------------
MODELO PARA HORIZONTE DE 3 DIAS
--------------------------------------------------
Accuracy: 0.5322
Precision (weighted): 0.5331
Recall (weighted): 0.5322
F1-Score (weighted): 0.5317
Precision (macro): 0.5329
Recall (macro): 0.5327
F1-Score (macro): 0.5319
Precision (Alta): 0.5437
Recall (Alta): 0.5000
F1-Score (Alta): 0.5210
Support: 342 amostras

Relatório detalhado por classe:
               precision    recall  f1-score   support

Baixa/Estável       0.52      0.57      0.54       168
         Alta       0.54      0.50      0.52       174

     accuracy                           0.53       342
    macro avg       0.53      0.53      0.53       342
 weighted avg       0.53      0.53      0.53       342


--------------------------------------------------
MODELO PARA HORIZONTE DE 7 DIAS
--------------------------------------------------
Accuracy: 0.5643
Precision (weighted): 0.5648
Rec

## 5. Análise Comparativa e Métricas Adicionais

In [9]:
# Comparação das métricas entre modelos com e sem dummy_period
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("COMPARAÇÃO DAS MÉTRICAS ENTRE MODELOS")
print("="*80)

# Criando DataFrame comparativo
comparison_data = []

for horizon in horizons:
    # Dados com dummy
    with_dummy = results_with_dummy[horizon]
    
    # Dados sem dummy
    without_dummy = results_without_dummy[horizon]
    
    comparison_data.append({
        'Horizonte': f"{horizon[:-1]} dias",
        'Modelo': 'Com Dummy',
        'Accuracy': with_dummy['accuracy'],
        'Precision (weighted)': with_dummy['precision'],
        'Recall (weighted)': with_dummy['recall'],
        'F1-Score (weighted)': with_dummy['f1_score']
    })
    
    comparison_data.append({
        'Horizonte': f"{horizon[:-1]} dias",
        'Modelo': 'Sem Dummy',
        'Accuracy': without_dummy['accuracy'],
        'Precision (weighted)': without_dummy['precision_weighted'],
        'Recall (weighted)': without_dummy['recall_weighted'],
        'F1-Score (weighted)': without_dummy['f1_score_weighted']
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nComparação das métricas principais:")
print(comparison_df.round(4))

print("\n" + "="*80)
print("MÉTRICAS ESPECÍFICAS PARA CLASSIFICAÇÃO BINÁRIA (Modelo sem Dummy)")
print("="*80)

# Tabela específica para métricas de classificação binária
binary_metrics = []
for horizon in horizons:
    metrics = results_without_dummy[horizon]
    binary_metrics.append({
        'Horizonte': f"{horizon[:-1]} dias",
        'Precision (Macro)': metrics['precision_macro'],
        'Recall (Macro)': metrics['recall_macro'],
        'F1-Score (Macro)': metrics['f1_score_macro'],
        'Precision (Alta)': metrics['precision_alta'],
        'Recall (Alta)': metrics['recall_alta'],
        'F1-Score (Alta)': metrics['f1_score_alta']
    })

binary_df = pd.DataFrame(binary_metrics)
print("\nMétricas específicas por classe:")
print(binary_df.round(4))

COMPARAÇÃO DAS MÉTRICAS ENTRE MODELOS

Comparação das métricas principais:
  Horizonte     Modelo  Accuracy  Precision (weighted)  Recall (weighted)  \
0    3 dias  Com Dummy    0.5263                0.5273             0.5263   
1    3 dias  Sem Dummy    0.5322                0.5331             0.5322   
2    7 dias  Com Dummy    0.5731                0.5761             0.5731   
3    7 dias  Sem Dummy    0.5643                0.5648             0.5643   
4   15 dias  Com Dummy    0.5965                0.5945             0.5965   
5   15 dias  Sem Dummy    0.5789                0.5801             0.5789   
6   30 dias  Com Dummy    0.6023                0.6048             0.6023   
7   30 dias  Sem Dummy    0.5585                0.5649             0.5585   

   F1-Score (weighted)  
0               0.5257  
1               0.5317  
2               0.5733  
3               0.5645  
4               0.5938  
5               0.5793  
6               0.6032  
7               0.5600  

MÉTRI

### Interpretação das Métricas para Modelos Financeiros

**Métricas Weighted vs Macro:**
- **Weighted**: Pondera pelas frequências das classes (útil quando há desbalanceamento)
- **Macro**: Média simples entre classes (trata ambas as classes igualmente)

**Métricas Específicas para Classe "Alta":**
- **Precision (Alta)**: Entre todas as predições de "alta", quantas estavam corretas?
- **Recall (Alta)**: Entre todos os casos reais de "alta", quantos foram identificados?
- **F1-Score (Alta)**: Harmônica entre precision e recall da classe "Alta"

**Para análise de mercado financeiro, considere também:**
1. **AUC-ROC**: Capacidade de discriminação entre classes
2. **Matriz de Confusão**: Análise detalhada de erros
3. **Validação Temporal**: Split por data (não aleatório) para séries temporais
4. **Métricas de Retorno**: Lucro/prejuízo baseado nas predições